# Overview
---

Purpose of fine-tuning = update model weights to follow specific type of instructions

__Full fine-tuning__ = we update all model weights<br>Same architecture, same loss<br>Larger learning rates, smaller batches, smaller dataset sizes.

Problem 1 = not efficient - intrinsic dimensionality of fine-tuning is usually much lower<br>
Problem 2 = "catastrphic forgetting" - a phenomena known from 1989 (g (McCloskey & Cohen) when training on new data rewrites the previously trained knowledge

Solution = __PEFT__ (parameter efficient fine-tuning)

__Idea:__ let's freeze most of the model weights $W$ and train only a shift vector $\Delta W$ which can be represented witha significantly smaller amount of parameters

Primary purpose of PEFT = memory reduction. Forward and backward<br>Inference compute is the same or higher

Implementation details to be considered (see MAM adapters, they explore this variability):
1) which parameters are approximated
    - self-attention matrices (there are 4 of them)
    - FFN matrices (there are 2 of them)
    - other weird stuff: normalization layers etc
2) commutation type
    - input-parallel = computed in a branch parallel to the frozen backbone
    - output-parallel = computed in a branch that follows the output of (should be called sequential but PEFT terminology is strange)
3) output fusion
    - additive (sumed with original signal)
    - via gating (weighted average with original signal)
    - substitution (aka fully sequential PEFT)
4) schedule
    - static
    - dynamic
5) unifromness
    - network-wise
    - layer-wise

Strategies to fight catastrophic forgetting are not limited to PEFT:<br>
- Mixin "general purpose" data with task specific data
    - __replay__ = mix old and new tasks in a dataset<br>different input/output format might be a problem, though solvable
    - do syntethic tasks on new data
    - dynamic interleaving = gradually shift to new instructions<br>
- Regularization:
    - Elastic Weight Consolidation: add parameter-specific penalty (via Fisher matrix)
    - use L2 penalties for parameter shift
    - Knowledge Distillation: penalize too large shifts from the answers [and parameters?] of original model<br>

# Adapters (2019)
---
[[paper]](https://arxiv.org/pdf/1902.00751)

One of the first works on efficient fine-tuning, often refered to as Houlsby

__Idea:__ let's enhance the network with some subnetwork (adapter) that would be fine-tuned on specific task and would change the behavior of the original network

They suggest adding 2 adapters to each layer (after self-attention modeule, after FFN module). Both are simply <u>added</u> to the original signal with some <u>scaling factor</u> s<br>During training original weights are freezed, only adapters are updated

<img src="img/adapters_houlsby.png" width=750>

Adapter = two-layer subnetwork<br>
$
h_{\text{out}} = h + s \cdot \left(W_{\text{up}}\,\phi\!\left(W_{\text{down}}\,h + b_{\text{down}}\right) + b_{\text{up}}\right)
$

Notice adapters have very similar to LoRA <u>bottleneck effect</u> = downscale projection $W_{up}$ is followed by upscale projection $W_{down}$<br>

The plot below shows the drop in performance depending on the amount of parameters being trained

<img src="img/adapters_performance.png" width=350>

Notice there is another popular variation of the method where adapters are applied only once after self-attention<br>
[[paper]](https://arxiv.org/abs/2004.03829)

# LoRA (2021)
---
[[paper]](https://arxiv.org/abs/2106.09685)

__Idea:__ instead of training the full scale update $\Delta W$ let's represent it as a product of two smaller weight matrices $W_{down} \cdot W_{up}$

There are 6 matrix transformeations in Transformers:
- $W_k, W_q, W_v, W_o$ in self-attention block
- $W_{down}, W_{up}$ in FFN block

Any of them can be decomposed (like in Hugginface implementation), but in the original paper the authors focus on two matrices $W_k, W_v$.

# Other LoRA family
---

### KroNA
[[paper]](https://arxiv.org/abs/2212.10650)

__Idea:__ instead of modeling dot-product we will model Kronecker product. Motivation = Kronecker product is by definition more compression-efficient 

$
A \otimes B =
\begin{bmatrix}
a_{11}B & a_{12}B & \cdots & a_{1n}B \\
a_{21}B & a_{22}B & \cdots & a_{2n}B \\
\vdots  & \vdots  & \ddots & \vdots  \\
a_{m1}B & a_{m2}B & \cdots & a_{mn}B
\end{bmatrix}
$

Basic property of standard matrix multiplcation:
$
\operatorname{rank}(XY) \le \min\big(\operatorname{rank}(X), \operatorname{rank}(Y)\big)
$
So the rank is at most $R$

Key property of Kronecker matrix multiplcation:
$
\operatorname{rank}(A \otimes B) = \operatorname{rank}(A) \cdot \operatorname{rank}(B)
$
So the rank is multiplcative



### DoRA (2022)
[[paper]](https://arxiv.org/abs/2402.09353)

DORA stands for "Weight-Decomposed LowRank Adaptation"

__Idea:__ instead of modeling dot-product we will model normalization decomposiotion - we represent weight matrix as a product of  <u>Direction</u> (per-row normalized version of the matrix) and <u>Magnitude</u> (per-rpw norms vector as a scaling factor)<br>
$W_{new}=W^{normed}_{old} \cdot diag(g)$

We model both components: the update $\Delta V$ and magnitude $r$
$W_{new}=(V_{old} + \Delta V) \cdot diag(g) = (V_{old} + \alpha V_A V_B) \cdot diag(g)$

Start with $W_0 = \frac{W_0}{||W_0||_{row}}$ and $g=||W_0||$

The same idea was used in some algorithms including FaceNorm.

### GLoRA
[[paper]](https://arxiv.org/pdf/2306.07967)

GLoRA = Generalized LoRA 

__Idea:__ let's derive general formula representation for methods

$
f(x) = (W_0 + W_0 A + B)\,x + C W_0 + D b_0 + E + b_0
$

Some methods they refer to in their research:
- SST = Shift and Scale Tuning (2022)
- RepAdapter
- VisionAdapter
- AdaptFormer
- FacT
- NOAH

Also they propose to perform NAS (network architecture search) to find optinmal combination

Let's derive formula for adapters
$
y = f(x) + s \cdot \left(W_{\text{up}}\,\phi\!\left(W_{\text{down}}\,f(x) + b_{\text{down}}\right) + b_{\text{up}}\right)
$

Here is the list of method implementations:

| Method                                            | $A$                                                     | $B$                                     | $C$                   | $D$              | $E$                                                     | Effect                                    |
| ------------------------------------------------- | ------------------------------------------------------- | --------------------------------------- | --------------------- | ---------------- | ------------------------------------------------------- | ----------------------------------------- |
| **LoRA**                                          | Low-rank matrix ($BA$ form), usually replacing $B$ term | $B$ = low-rank update $\Delta W$, $A=0$ | 0                     | 0                | 0                                                       | Weight update only                        |
| **SSF (Scaling and Shifting Features)**           | 0                                                       | 0                                       | 0                     | Vector scale $D$ | Vector shift $E$                                        | Scales/shifts activations per feature     |
| **Adapters** (Houlsby-style)                      | Full small bottleneck in $B$                            | $B$ = up-proj∘down-proj                 | 0                     | 0                | 0                                                       | Adds trainable transformation in parallel |
| **VPT-Deep** (Visual Prompt Tuning, deep variant) | 0                                                       | 0                                       | Prompt vectors in $C$ | 0                | 0                                                       | Injects learned prompts at each layer     |
| **BitFit**                                        | 0                                                       | 0                                       | 0                     | 0                | Train only bias $b_0$ ⇒ set $E$ trainable, others fixed | Bias-only fine-tuning                     |
| **Combined methods** (e.g., LoRA + SSF)           | Low-rank $B$                                            | + $D, E$ nonzero                        | Optional              | Optional         | Optional                                                | Weight LoRA + activation scaling          |



### GaLoRe (2024)

[[paper]](https://arxiv.org/abs/2403.03507v1)

__Idea:__ let's model gradient matrix update instead of weight matrix update

### AdaLoRA (2023)

[[paper]](https://arxiv.org/abs/2303.10512)

__Idea:__ let's adjust rank r while training the model

### QLoRA

__Idea:__ quantize the freezed model to 4-bit

# FacT
---

# NOAH
---


# Prompt tuning (2021)
---
[[paper]](https://arxiv.org/pdf/2104.08691)

__Idea:__ enforce change in model prediction by prepending task specific tokens to the input

<img src="img/prompt_tuning.png" width=350>

\# of trained params: $p \text{prefix size} \times d$

Multiple task examples can be mixed in one batch

# Prefix tuning (2021)
---
[[paper]](https://arxiv.org/abs/2101.00190)

__Idea:__ prepend task specific constants to <u>ALL</u> embeddings not only the input ones

Trained params: $L \text{(layers)} \times p \text{(prefix size)} \times 2 \text{(usually K and V are enhanced)} \times d \text{(model dimensionality)}$

Prefix tuning performs better than prompt tuning and usually is default. Prompt tuning may be benefitial when you are short on resources

__NOTE:__ both prompt tuning and prefix tuning are usually applied to self-attention matrices and rarely to FFN matrices

# Ladder Side Tuning (2022)
---
[[paper]](https://arxiv.org/abs/2206.06522)

__Idea:__ let's construct a separate signal highway in parallel to the main pipeline, that wil be used to model shift vector. 

This construction is made to be totally universal - each layer can be configured to consume signal from any number of <u>previous</u> layers $i \rightarrow j$

<img src="img/lst.png" width=350>

The method gets its name from the network visualisation because it looks like a ladder

# ${\text{IA}}^3$ (2022)
---
[[paper]](https://arxiv.org/abs/2205.05638)

IA = Identity Adapter<br>

__Idea:__ let's add a linear addon to each transformer layer output. Its purpose is to 
Adapter is a 2 layer network with nonlinear activation in-between

$h' = T(h) + W_1 \cdot f(W_2h)$

### Comparison to ICL
- First, processing all prompted input-target pairs every time the model makes a prediction incurs significant compute costs.
- Second, ICL typically produces inferior performance compared to fine-tuning [4].
- Finally, the exact formatting of the prompt (including the wording [11] and ordering of examples [12]) can have significant and
unpredictable impact on the model’s performance, far beyond inter-run variation of fine-tuning. Recent work has also demonstrated that ICL can perform well even when provided with incorrect labels, raising questions as to how much learning is taking place at all [9]


# Bitfit (2021)
---
[[paper]](https://arxiv.org/abs/2106.10199)

__Idea:__ Freeze all weights except the biases in all linear layers<br>
Updates only 0.05% weights, which is eneough for small models, but not enough for large models

# DiffPruning (2020)
---
[[paper]](https://arxiv.org/abs/2012.07463)

__Idea:__ let's learn task-specific difference vector $W_{SFT} \leftarrow W_{pretrain} + \Delta W$. Training efficiency is achieved by enforcing $L_0$ sparsity of $\Delta W$ since we assume not all weights are eqaully useful

Note that they use $L_0$ not $L_1$, which complicates things a bit<br>[Not sure why, probably $L_1$ does not guarantee zeroing out. But why not binarizing?]<br>
\*$L_0$ = number of non-zero weights

They refer to the work where the authors explored the ways of $L_0$ optimization, called "differentiable approximation to the $L_0$ norm" (2018)<br>
[[paper]](https://arxiv.org/abs/1712.01312)

For each weight $w_i$ there is also a companion gating score $z_i$ being computed, which clamps the weigth to either zero or one

$w_{\text{masked}} \leftarrow w \odot \tilde{z}$

Such binarization is modeled by a (partially differentiable) __HardSigmoid__ function. Here $\beta$ - smoothness parameter, $\gamma$,  $\zeta$ - transition area, $\alpha_i$ gating score for weight

$u \sim \text{Uniform}(0,1)$ - some random noise, required for exploration

$\tilde{z}_i = \text{HardSigmoid}\left( \frac{\log u - \log (1-u) + \alpha_i}{\beta} \cdot (\zeta - \gamma) + \gamma \right)$ - gating score regulated by $\alpha$ parameter, for important weights $\alpha$'s set to be higher

The overall loss is

$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{SFT}} + \mathcal{L}_{\text{sparsity}} = \mathcal{L}_{\text{SFT}} + \lambda \sum_{i} P(z_i \neq 0)$


# FAR (2022)
---
[[paper]](https://arxiv.org/pdf/2205.01541)

FAR stands for "Freeze and Reconfigure"

Very similar to LST = each layer's input is a linear combination of previous layers outputs<br>
BUT in LST previous outputs are simply summed (model learns which layers to use), in FAR method we fuse <u>ALL</u> previous layers with different weights - model leans how to mix

Important = no additional output processing and no side networks are trained, only mix parameters are trained

# FishMask (2025)
---
[[paper]](https://arxiv.org/pdf/2504.04050)

For each task create a binary mask of top-K important weights according to Fisher Information metric $P(y|x)$<br>


# DiffFit 
---
Idea: combine BitFit and DiffPruning. We train biases in all linear layers and 

# Unipelf (2021)
---
[[paper]](https://arxiv.org/abs/2110.07577)

__Idea:__ let's use all three addons (adapters, prefix, LoRA) at once. Their results will be fused using trained <u>gating</u> weights

<img src="img/unipelt.png" width=300>

# Intrinsic SAID
---
[[paper]]()

__Idea:__ map a random small vector to $W$

Fastfood transform is a complicated linear transformation $F(w_r) = \frac{1}{\sigma \sqrt{n}}BH\Pi GS$

where
- B – diagonal matrix with random ±1 entries
- H – Walsh–Hadamard matrix (fast 
- Π – random permutation matrix
- G – diagonal Gaussian random variables
- S – scaling factors for variance control
- σ – normalization constant

It maps $R^d \rightarrow R^N$

# MAM Adapters (2021)
---
[[paper]](https://arxiv.org/pdf/2110.04366)

The authors explore 3 existing approaches to fine-tuning and give a good overview of them: 
- __Adapters__<br>you model the shift from parameter vector $\Delta W$ not the vector itself $W$<br>
- __Prefix tuning__<br>we encode the type of task that would change the prediction behavior as a prepending token embedding
- __Reparameterization__<br>LoRA = we model weight update as an approximation $\Delta W = W_{down} \cdot W_{up}$

*Prompt tuning = similar to Prefix tunining but applied to input layer only

<img src="img/peft_types.png" width=500>

Questions posed by authors:
1. How are these methods connected?
2. Do these methods share design elements that are essential for their effectiveness, and what are they? 
3. Can the effective ingredients of each method be transferred to others to yield more effective variants?

And though they dont give the answer to any of those, they make us notice that these methods are indeed similar

$h \leftarrow h + f(x W_{down})W_{up}$

$head_i \leftarrow Attention()$

$h \leftarrow h + s x W_{down} W_{up}$

PLM module = either Self-attention or FFN

Under proper inspection we can see that all three strategies are quite similar

The authors decided to describe each method in terms of 4 key characteristics:
- __Functional form__<br>how exactly each add-on is computed
- __Modified representation__<br>which layer is being directly affected
- __Insertion form__:
    - sequential: after PLM module
    - parallel: in parallel with PLM module
- __Composition function__<br>how add-ons are fused into main signal

<img src="img/peft_table.png" width=750>

This characterisation made possible to introduce 3 new versions:
1) __Parallel Adapter__<br>apply Adapter in parallel
2) __Multi-head Parallel Adapter__<br>apply parallel adapters for multiple attention heads
3) __Scaled Parallel Adapter__<br>parallel adapter that borrows composition and insertion form of LoRA into adapters

Adapters can be inserted in Self-attention/FFN while prefix tunining only to attention
Adapters are single-headed, while prefixes can be applied to each attention head

1. Do methods varying the design elements above exhibit distinct properties? 
2. Which design dimensions are particularly important? 
3. Do the novel methods described above yield better performance?

Experiment setup:<br>
they explored 2 pre-trained models BART and RoBERTa on 4 fine-tuning tasks SUM (summarization), Translation, MNLI (language inference), SST (sentiment)

Findings:
1) Scaled parallel adapter is the best variant
to modify FFN;
2) FFN can better utilize modification at larger capacities; and
3) modifying head attentions like prefix tuning can achieve strong performance with only 0.1% parameters

We mix and match the favorable designs behind these findings: specifically, we use prefix
tuning with a small bottleneck dimension (l = 30) at the attention sub-layers and allocate more
parameter budgets to modify FFN representation using the scaled parallel adapter (r = 512). Since
prefix tuning can be viewed as a form of adapter in our unified framework, we name this variant
as Mix-And-Match adapter (MAM Adapter)


# Compacter (2021)
---
[[paper]](https://arxiv.org/abs/2106.04647?utm_source=chatgpt.com)




# Attention Fusion (2021)
---
[[paper]](https://arxiv.org/pdf/2005.00247)

__Idea:__ train one or several adapters, but instead of simple summing let's combine their output with additional <u>attention layer</u>

Some terminology regarding Fine-Tuning practices:
- MTL (multi-task learning) = minimize all task-specific losses in the same training loop<br>$L = \sum_{i=1}^N L_i (D_i, \Theta_0)$
- MTA (multi-task adapters) = minimize all task-specific adapters in parallel
- STA (single task adapter)

Algorithm:
1) freeze backbone and train all adapters
2) freeze backbone and adapters and train a new "fusion" layer

<img src="img/adapterfusion2.png" width=250>

Adapter location is not precisely specified, but they refer to Houlsby - we assume two locations

How signal is fused?
- via additional attention layer (K, Q, V)
- Q is taken from backbone signal
- K, V are a signal from adapers

<img src="img/adapterfusion.png" width=240>




# S4
---
[[paper]](https://arxiv.org/abs/2111.00396)



__Intrinsic dimensionality (ID)__ refers to the effective number of degrees of freedom needed to describe the variation in a dataset, a learned representation, or a model’s parameter space. The number of meaningful dimensions that actually influence outputs in a significant way
- For data = the dimension of the manifold the data lies on inside the huge input space
- For models = minimum number of parameters you actually need to fit a function as well as a big model would

Experiment<br>select K downstream tasks and sweep over different LoRA dimensions. Model saturates if performance drop is less than 1%

Ratio of Weights Needed for Task Adaptation:
- LoRA typical ranks:<br>
Rank 4–64 updates correspond to ~0.01% – 1% of total parameters being trained.

- BitFit (bias-only tuning):<br>
Updates <0.1% of parameters.

- Adapter layers:<br>
Updates ~1–3% of parameters.

- Prefix-tuning:<br>
Equivalent to <0.1% of parameters.

# Compression Ratio
| Model                  | Full Parameters            | Typical Fine-Tuning Method | Params Updated | Ratio Updated            | Notes                                            |
| ---------------------- | -------------------------- | -------------------------- | -------------- | ------------------------ | ------------------------------------------------ |
| **BERT-Base**          | 110M                       | LoRA (rank=8)              | \~0.09M        | 0.08%                    | Classic benchmark for ID studies.                |
| **RoBERTa-Large**      | 355M                       | LoRA (rank=16)             | \~0.6M         | 0.17%                    | Still tiny fraction needed.                      |
| **GPT-2 Medium**       | 345M                       | Prefix-tuning (20 tokens)  | \~0.2M         | 0.06%                    | Comparable to LoRA performance.                  |
| **GPT-2 XL**           | 1.5B                       | LoRA (rank=16)             | \~2.6M         | 0.17%                    | Matches full fine-tuning on many NLP benchmarks. |
| **LLaMA-7B**           | 7B                         | LoRA (rank=64)             | \~9M           | 0.13%                    | Common config for instruction tuning.            |
| **LLaMA-13B**          | 13B                        | LoRA (rank=64)             | \~17M          | 0.13%                    | Same ratio scaling as 7B.                        |
| **LLaMA-65B**          | 65B                        | LoRA (rank=64)             | \~85M          | 0.13%                    | Used in PEFT studies; huge savings.              |
| **Mistral-7B**         | 7B                         | LoRA (rank=16)             | \~2.3M         | 0.03%                    | Good trade-off for inference speed.              |
| **Falcon-40B**         | 40B                        | LoRA (rank=16)             | \~13M          | 0.03%                    | Popular for domain adaptation.                   |
| **Mixtral-8×7B (MoE)** | 47B total (active \~12.9B) | LoRA (rank=16)             | \~4.2M         | 0.03% of *active* params | Only active experts get LoRA adapters.           |

# Review of the methods
---
| Method              | Placement                     | Computation          | Fusion                                    | Static/Dynamic | Param style                            | Mergeable?            | Granularity                    |
| ------------------- | ----------------------------- | -------------------- | ----------------------------------------- | -------------- | -------------------------------------- | --------------------- | ------------------------------ |
| **LoRA**            | Q,K,V,O, FFN in/out           | Parallel             | Additive                                  | Static         | Weight reparam (low-rank)              | Yes                   | Uniform or heterogeneous       |
| **DoRA**            | Same as LoRA                  | Parallel             | Additive                                  | Static         | Weight reparam (norm × direction)      | Yes                   | Uniform/heterogeneous          |
| **Houlsby Adapter** | After SA & FFN sublayers      | Parallel             | Additive                                  | Static         | Weight reparam (bottleneck MLP)        | Sometimes (if linear) | Uniform                        |
| **BitFit**          | All biases                    | Parallel (bias path) | Additive                                  | Static         | Weight reparam (bias only)             | Yes                   | Uniform                        |
| **SSF**             | After attention & FFN outputs | Parallel             | Multiplicative (scale) + Additive (shift) | Static         | Activation modulation                  | No                    | Uniform                        |
| **VPT-Deep**        | Input & after each block      | Sequential           | Additive (concat in sequence)             | Static         | Prompt injection                       | No                    | Uniform                        |
| **Compacter**       | Q,K,V,O, FFN                  | Parallel             | Additive                                  | Static         | Weight reparam (Kronecker low-rank)    | Yes                   | Uniform                        |
| **AdaLoRA**         | Q,K,V,O, FFN                  | Parallel             | Additive                                  | Static         | Weight reparam (low-rank)              | Yes                   | Heterogeneous (adaptive ranks) |
| **MoE-LoRA**        | Q,K,V,O                       | Parallel             | Gating (input-dependent)                  | Dynamic        | Weight reparam (low-rank)              | No                    | Uniform                        |
| **MAM Adapter**     | After SA & FFN sublayers      | Sequential           | Gating (dynamic mask)                     | Dynamic        | Activation modulation + weight reparam | No                    | Heterogeneous                  |
